In [ ]:
# get a combined product example

from pathlib import Path
from datetime import date

from wekeo_combined_chain.combined import get_combined_product

s5p_pca_test_file = Path("/mnt/ceph/proj/WEKEO/S5P_PCA_V0.1") / "S5P-PCA-BD7_2021-08-06_v0.1.nc"

# day = date(2021, 8, 1)

# explicit call example
ds = get_combined_product(
    s5p_pca_product=s5p_pca_test_file,
    plumes=True,
    frp_slstr_l3=True,
    iasi_l3=True,
    width=3272,
    save_result=False,
    use_cache=False,
    use_cache_subchains=True,
)


In [ ]:
from wekeo_combined_chain import postprocess

# Stage 1 — per-plume statistics
df_plumes = postprocess.compute_plume_stats(ds)
df_plumes

In [ ]:
# Quick look at columns available
print(df_plumes.shape)
df_plumes[["label", "centroid_lat_plume", "centroid_lon_plume",
           "frp_energy_SWIR_plume", "frp_energy_MWIR_plume",
           "fire_score_SWIR_plume", "fire_score_MWIR_plume",
           "source_confidence_label_SWIR_plume", "source_confidence_label_MWIR_plume"]].head(10)

In [ ]:
# Optionally inspect / filter before building grids
# e.g. keep only plumes with confirmed SWIR fire
df_confirmed = df_plumes[df_plumes["frp_energy_SWIR_plume"].notna() & (df_plumes["frp_energy_SWIR_plume"] > 0)]
print(f"{len(df_confirmed)} / {len(df_plumes)} plumes with SWIR FRP")

In [ ]:
# Stage 2 — build 2-D gridded output from the plume stats
ds_post = postprocess.build_grids(ds, df_plumes)
ds_post

In [ ]:
# Save outputs
output_dir = Path("./output")
output_dir.mkdir(exist_ok=True)

df_plumes.to_csv(output_dir / "plumes_summary.csv", index=False, float_format="%.4f")
ds_post.to_netcdf(output_dir / "postprocess_output.nc")
print("Saved plumes_summary.csv and postprocess_output.nc")

In [ ]:
# Generate all maps (requires a date string for titles/filenames)
date_str = "20210806"

postprocess.plot(
    ds_combined=ds,
    ds_post=ds_post,
    df_plumes=df_plumes,
    output_dir=str(output_dir / "plots"),
    date_str=date_str,
)

In [ ]:
# --- Alternative: one-shot convenience wrapper ---
# ds_post, df_plumes = postprocess.compute(ds)